# Data Exploration: Dysarthria Dataset

This notebook explores the dysarthria dataset:
- Dataset statistics and class distribution
- Audio characteristics (duration, sample rate, quality)
- Label analysis
- Sample audio playback

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import librosa
import librosa.display
from IPython.display import Audio, display

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load Manifests

In [ ]:
# Load train/val/test manifests
train_df = pd.read_csv('../data/manifests/train.csv')
val_df = pd.read_csv('../data/manifests/val.csv')
test_df = pd.read_csv('../data/manifests/test.csv')

print(f"Train samples: {len(train_df)}")
print(f"Val samples:   {len(val_df)}")
print(f"Test samples:  {len(test_df)}")
print(f"Total:         {len(train_df) + len(val_df) + len(test_df)}")

In [ ]:
# Preview train manifest
train_df.head()

## 2. Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, df, title in zip(axes, [train_df, val_df, test_df], ['Train', 'Val', 'Test']):
    counts = df['label'].value_counts()
    ax.bar(['Healthy', 'Dysarthric'], [counts.get(0, 0), counts.get(1, 0)], color=['#2ecc71', '#e74c3c'])
    ax.set_title(f'{title} Set', fontsize=14, fontweight='bold')
    ax.set_ylabel('Count')
    ax.grid(axis='y', alpha=0.3)
    
    # Add percentages
    total = len(df)
    for i, count in enumerate([counts.get(0, 0), counts.get(1, 0)]):
        pct = count / total * 100
        ax.text(i, count + 5, f'{pct:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Audio Duration Analysis

In [ ]:
# Load a sample to compute durations (if not in manifest)
def get_audio_duration(file_path):
    try:
        y, sr = librosa.load(file_path, sr=None)
        return len(y) / sr
    except:
        return None

# Sample 100 files for duration analysis
sample_files = train_df.sample(n=min(100, len(train_df)))
durations = [get_audio_duration(f) for f in sample_files['file_path']]
durations = [d for d in durations if d is not None]

print(f"Mean duration: {np.mean(durations):.2f}s")
print(f"Median duration: {np.median(durations):.2f}s")
print(f"Min duration: {np.min(durations):.2f}s")
print(f"Max duration: {np.max(durations):.2f}s")

In [ ]:
plt.figure(figsize=(12, 5))
plt.hist(durations, bins=30, edgecolor='black', alpha=0.7)
plt.axvline(np.mean(durations), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(durations):.2f}s')
plt.xlabel('Duration (seconds)', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.title('Audio Duration Distribution', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.show()

## 4. Listen to Sample Audio

In [ ]:
# Sample healthy audio
healthy_sample = train_df[train_df['label'] == 0].sample(n=1).iloc[0]
print(f"Healthy sample: {healthy_sample['file_path']}")
y_healthy, sr_healthy = librosa.load(healthy_sample['file_path'], sr=16000)
display(Audio(y_healthy, rate=sr_healthy))

In [ ]:
# Sample dysarthric audio
dysarthric_sample = train_df[train_df['label'] == 1].sample(n=1).iloc[0]
print(f"Dysarthric sample: {dysarthric_sample['file_path']}")
y_dysarthric, sr_dysarthric = librosa.load(dysarthric_sample['file_path'], sr=16000)
display(Audio(y_dysarthric, rate=sr_dysarthric))

## 5. Visualize Waveforms and Spectrograms

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Healthy waveform
librosa.display.waveshow(y_healthy, sr=sr_healthy, ax=axes[0, 0])
axes[0, 0].set_title('Healthy - Waveform', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Time (s)')
axes[0, 0].set_ylabel('Amplitude')

# Healthy spectrogram
D_healthy = librosa.amplitude_to_db(np.abs(librosa.stft(y_healthy)), ref=np.max)
librosa.display.specshow(D_healthy, sr=sr_healthy, x_axis='time', y_axis='hz', ax=axes[0, 1], cmap='viridis')
axes[0, 1].set_title('Healthy - Spectrogram', fontsize=14, fontweight='bold')
axes[0, 1].set_ylabel('Frequency (Hz)')

# Dysarthric waveform
librosa.display.waveshow(y_dysarthric, sr=sr_dysarthric, ax=axes[1, 0])
axes[1, 0].set_title('Dysarthric - Waveform', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Time (s)')
axes[1, 0].set_ylabel('Amplitude')

# Dysarthric spectrogram
D_dysarthric = librosa.amplitude_to_db(np.abs(librosa.stft(y_dysarthric)), ref=np.max)
librosa.display.specshow(D_dysarthric, sr=sr_dysarthric, x_axis='time', y_axis='hz', ax=axes[1, 1], cmap='viridis')
axes[1, 1].set_title('Dysarthric - Spectrogram', fontsize=14, fontweight='bold')
axes[1, 1].set_ylabel('Frequency (Hz)')

plt.tight_layout()
plt.show()

## 6. Summary Statistics

In [ ]:
summary = pd.DataFrame({
    'Split': ['Train', 'Val', 'Test', 'Total'],
    'Total': [len(train_df), len(val_df), len(test_df), len(train_df) + len(val_df) + len(test_df)],
    'Healthy': [
        (train_df['label'] == 0).sum(),
        (val_df['label'] == 0).sum(),
        (test_df['label'] == 0).sum(),
        (train_df['label'] == 0).sum() + (val_df['label'] == 0).sum() + (test_df['label'] == 0).sum(),
    ],
    'Dysarthric': [
        (train_df['label'] == 1).sum(),
        (val_df['label'] == 1).sum(),
        (test_df['label'] == 1).sum(),
        (train_df['label'] == 1).sum() + (val_df['label'] == 1).sum() + (test_df['label'] == 1).sum(),
    ],
})

summary['Healthy %'] = (summary['Healthy'] / summary['Total'] * 100).round(1)
summary['Dysarthric %'] = (summary['Dysarthric'] / summary['Total'] * 100).round(1)

summary